# Experiment: Biohub detection density and count calibration

Objective: determine whether per-movie detection counts are systematically under-called, and whether a conservative count target can improve the node-count multiplier without damaging edge Jaccard.
The first pass is diagnostic-only: it records predicted nodes, estimated true nodes, node recall, edge metrics, and the implied count multiplier before any calibration changes.


In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

import random
import statistics
from pathlib import Path
import pandas as pd

SEED = 7
random.seed(SEED)
SEED


## Plan

- Hypothesis: the `44b6_0b24845f` movie is materially under-detected, while the two dense `6bba` movies are closer to their estimated counts.
- Variables to sweep later: point threshold, physical peak-pool radius, and a per-movie target count multiplier; never alter the linker until the count-only effect is measured.
- Metrics: predicted node count, estimated true node count, node recall, edge TP/FP/FN, adjusted edge Jaccard, and `(1 - 0.1*(N_pred-N_true)/N_true)`.


In [ ]:
# Known labelled-movie targets from the competition metadata and prior audit.
estimated_true_nodes = pd.DataFrame({
    "dataset": ["44b6_0113de3b", "44b6_0b24845f", "6bba_05b6850b", "6bba_05db0fb1", "44b6_33b596bf"],
    "estimated_true_nodes": [25755, 32795, 6362, 69800, 23330],
})
estimated_true_nodes["target_ratio"] = 1.0
estimated_true_nodes


## Results

- Compare the minimal direct-export graph against these counts before changing thresholds.
- A useful calibration must improve the under-counted movie without increasing false nodes on the two dense movies.
- Do not use the labelled split as a global optimizer: report per-movie results and preserve a frozen control.


In [ ]:
# Calibration proposal; this is intentionally not applied to predictions yet.
calibration_grid = pd.DataFrame({
    "target_ratio": [0.90, 0.95, 1.00, 1.05, 1.10],
    "decision": ["diagnostic"] * 5,
})
calibration_grid


## Next steps

- Add a local-mode export of per-movie node counts from the minimal ILP notebook.
- Sweep detector threshold and peak-pool radius independently on the under-counted movie.
- Only then test a bounded per-movie count target; reject any setting that changes edge precision materially.
